# Notebook 20 — Bayesian Optimization Hyperparameter Tuning (Optuna/TPE) untuk LSTM Model B

Notebook ini mencari kombinasi hyperparameter (terutama **ukuran hidden layer** —
`hidden_size` & `num_layers`) untuk `FlexibleLSTMClassifier` memakai **Bayesian
Optimization**, sebagai pembanding langsung terhadap **Random Search** yang dipakai
model resmi skripsi (`07_LSTM_FinBERT_skripsi_scoped.ipynb` / endpoint API
`training_service.py`).

**Yang di-reuse APA ADANYA dari pipeline resmi** (bukan reinvent, apples-to-apples):
- Data: `pipeline/feature_engineering/dataset_builder.py::load_return_features()` +
  `RETURN_FEATURE_COLS_B` (5 fitur *return* OHLCV + `sentiment_mean` — fitur stasioner,
  temuan Notebook 11/12) dengan `RETURN_HORIZON = 5` hari (`models/lstm/notebook07_trainer.py`
  — nilai resmi terkini, DEVIASI Batasan Masalah BAB I yang sudah disetujui user 2026-07-28).
- Split & preprocessing: `models/lstm/notebook07_common.py::prepare_dataset_bundle()` —
  split kronologis 70/15/15, capping outlier IQR (fit-train-only), `MinMaxScaler`
  (fit-train-only), windowing (`window=20`, sama seperti `notebook07_trainer.WINDOW`).
- Arsitektur: `FlexibleLSTMClassifier` (2-class softmax, aktivasi FC-head configurable)
  — SAMA yang dipakai Random Search resmi, TIDAK dibuat model baru.
- Search space (`hidden_size`, `num_layers`, `dropout`, `optimizer`, `activation`, `lr`,
  `batch_size`): PERSIS `models/lstm/notebook07_trainer.py::SEARCH_SPACE` — **satu-satunya
  beda** dari Random Search resmi adalah (1) teknik sampling konfigurasi (Bayesian vs
  uniform-random) dan (2) `epochs` diganti *early stopping* berbasis `val_f1_macro`
  (pelajaran Notebook 13: `val_loss` minimum di epoch awal lalu naik pada tugas ber-SNR
  rendah — early stopping HARUS memantau `val_f1_macro`, bukan `val_loss`).

**Apa itu Bayesian Optimization di sini:** dipakai `optuna.samplers.TPESampler`
(*Tree-structured Parzen Estimator*) — teknik *sequential model-based optimization*:
tiap trial baru dipilih dengan membangun model probabilistik (surrogate) dari hasil
trial-trial sebelumnya, lalu memakai fungsi akuisisi untuk menyeimbangkan
*eksplorasi* (area belum dicoba) vs *eksploitasi* (area yang sudah terbukti bagus) —
BUKAN menebak konfigurasi berikutnya secara acak/independen seperti Random Search.
`optuna.pruners.MedianPruner` dipakai untuk memangkas trial yang jelas tidak menjanjikan
lebih awal (hemat komputasi), berdasarkan `val_f1_macro` per-epoch.

**`N_TRIALS = 20`** — disamakan PERSIS dengan `N_ITER` Random Search resmi
(`notebook07_trainer.N_ITER`) supaya perbandingan murni karena TEKNIK pencarian,
bukan karena jumlah percobaan yang beda.

**Prinsip anti p-hacking (konsisten seluruh notebook project ini):** ini SATU kali
run Bayesian Optimization dibandingkan SATU kali run Random Search (yang sudah
tersimpan sebagai model resmi) — kedua proses stokastik. Kalau BO menang tipis,
itu BUKAN bukti kuat superioritas metodologis (butuh beberapa repetisi seed untuk
klaim statistik yang valid) — dilaporkan apa adanya di Bagian 7, bukan diklaim
berlebihan.

**Status: notebook ini ditulis LENGKAP tapi BELUM DIEKSEKUSI** (sesuai permintaan
user) — `N_TRIALS=20 x hingga MAX_EPOCHS=80` per trial bisa memakan waktu, jalankan
terpisah (disarankan headless via nbconvert/papermill, bukan cell-by-cell Jupyter MCP,
konsisten catatan token di `CLAUDE.md`).

In [ ]:
from __future__ import annotations

import json
import logging
import sys
from copy import deepcopy
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("E:/forecasting_index_snp500/stock_prediction")
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("notebook20")
optuna.logging.set_verbosity(optuna.logging.WARNING)  # jangan banjiri log per-trial internal Optuna

OUT_DIR = ROOT / "notebooks" / "outputs" / "notebook20"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Device: %s", device)
logger.info("Optuna version: %s", optuna.__version__)

from pipeline.feature_engineering.dataset_builder import RETURN_FEATURE_COLS_B, load_return_features
from models.lstm.lstm_model import FlexibleLSTMClassifier
from models.lstm.notebook07_common import (
    _train_one_epoch,
    build_optimizer,
    compute_class_weights,
    prepare_dataset_bundle,
    sweep_best_threshold,
)
from models.lstm.notebook07_trainer import (
    ARTIFACT_DIR,
    FINBERT_FILES,
    N_ITER,
    RETURN_HORIZON,
    SEARCH_SPACE,
    WINDOW,
)

print("FEATURE_COLS (Model B, return-based):", RETURN_FEATURE_COLS_B)
print("HORIZON:", RETURN_HORIZON, " WINDOW:", WINDOW, " N_ITER (Random Search resmi):", N_ITER)
print("SEARCH_SPACE:", json.dumps(SEARCH_SPACE, indent=2))

## Bagian 1 — Data (reuse `load_return_features()` produksi, split 70/15/15)

Sama persis pipeline resmi Model B: `load_return_features()` → `prepare_dataset_bundle()`
(capping IQR fit-train-only → `MinMaxScaler` fit-train-only → windowing). Split
**70/15/15** dipertahankan (BUKAN diganti 80/20) supaya test set yang dipakai evaluasi
akhir Bagian 6 IDENTIK dengan test set yang menghasilkan `finbert_lstm_skripsi_metadata.json`
resmi — perbandingan Bagian 7 baru valid kalau test set-nya sama.

In [ ]:
bundle = prepare_dataset_bundle(
    load_data_fn=load_return_features,
    capping_cols=RETURN_FEATURE_COLS_B,
    feature_cols=RETURN_FEATURE_COLS_B,
    horizon=RETURN_HORIZON,
    train_ratio=0.70,
    val_ratio=0.15,
    window=WINDOW,
)

print("Split rows   :", bundle.split_sizes)
print("X_train shape:", bundle.X_train.shape, " X_val shape:", bundle.X_val.shape, " X_test shape:", bundle.X_test.shape)
print(f"Up% train={bundle.y_train.mean():.1%}  val={bundle.y_val.mean():.1%}  test={bundle.y_test.mean():.1%}")

train_ds = TensorDataset(torch.from_numpy(bundle.X_train), torch.from_numpy(bundle.y_train))
X_val_t = torch.from_numpy(bundle.X_val).to(device)
class_weights = compute_class_weights(bundle.y_train, device)

## Bagian 2 — Fungsi Objective Bayesian Optimization

Tiap trial: sampling 1 konfigurasi dari `SEARCH_SPACE` (lewat Optuna, bukan
`sample_config()` Random Search) → latih `FlexibleLSTMClassifier` dengan *early
stopping* berbasis `val_f1_macro` (threshold di-*sweep* tiap epoch via
`sweep_best_threshold()` — resep Notebook 09/12 mencegah config yang *collapse* ke 1
kelas dipilih sebagai "terbaik"). `MIN_EPOCHS`/`PATIENCE`/`MAX_EPOCHS` mengikuti resep
Notebook 19 (replikasi perbaikan Notebook 13).

`trial.report()` + `trial.should_prune()` dipanggil tiap epoch — `MedianPruner`
menghentikan trial lebih awal kalau `val_f1_macro`-nya di bawah median trial lain pada
epoch yang sama (setelah `n_warmup_steps`), murni optimasi komputasi, TIDAK memengaruhi
kriteria seleksi konfigurasi terbaik.

State model terbaik tiap trial disimpan di `trial_artifacts` (dict Python biasa, di
luar Optuna) — supaya model terbaik lintas SEMUA trial bisa langsung dipakai di Bagian
5 tanpa retrain ulang.

In [ ]:
MAX_EPOCHS = 80   # batas atas "epochs" di SEARCH_SPACE resmi ([30, 50, 80])
MIN_EPOCHS = 20   # resep Notebook 19/13 -- jangan stop sebelum model sempat belajar
PATIENCE = 15
N_TRIALS = N_ITER  # 20 -- disamakan dgn Random Search resmi (perbandingan teknik, bukan budget)

trial_artifacts: dict[int, dict] = {}


def objective(trial: optuna.Trial) -> float:
    config = {
        "hidden_size": trial.suggest_categorical("hidden_size", SEARCH_SPACE["hidden_size"]),
        "num_layers": trial.suggest_categorical("num_layers", SEARCH_SPACE["num_layers"]),
        "dropout": trial.suggest_float("dropout", *SEARCH_SPACE["dropout"]),
        "optimizer": trial.suggest_categorical("optimizer", SEARCH_SPACE["optimizer"]),
        "activation": trial.suggest_categorical("activation", SEARCH_SPACE["activation"]),
        "lr": trial.suggest_float("lr", *SEARCH_SPACE["lr"], log=True),
        "batch_size": trial.suggest_categorical("batch_size", SEARCH_SPACE["batch_size"]),
    }

    torch.manual_seed(SEED + trial.number)
    model = FlexibleLSTMClassifier(
        input_size=len(RETURN_FEATURE_COLS_B),
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
        activation=config["activation"],
        num_classes=2,
    ).to(device)

    optimizer = build_optimizer(config["optimizer"], model.parameters(), config["lr"])
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)

    best_f1_macro, best_epoch, best_threshold, best_state = -1.0, 0, 0.5, None
    epochs_no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        _train_one_epoch(model, train_loader, optimizer, criterion, device)

        model.eval()
        with torch.no_grad():
            val_probs_up = torch.softmax(model(X_val_t), dim=1)[:, 1].cpu().numpy()
        val_threshold, val_f1_macro = sweep_best_threshold(bundle.y_val, val_probs_up)

        trial.report(val_f1_macro, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_f1_macro > best_f1_macro + 1e-4:
            best_f1_macro, best_epoch, best_threshold = val_f1_macro, epoch, val_threshold
            best_state = deepcopy({k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            epochs_no_improve = 0
        elif epoch >= MIN_EPOCHS:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                break

    trial_artifacts[trial.number] = {
        "state_dict": best_state, "best_epoch": best_epoch,
        "threshold": best_threshold, "config": config,
    }
    return best_f1_macro

## Bagian 3 — Jalankan Bayesian Optimization (Optuna, `TPESampler`)

`sampler=TPESampler(seed=SEED)` untuk reproducibility. `direction="maximize"` karena
metrik seleksi adalah `val_f1_macro` (bukan loss).

In [ ]:
sampler = optuna.samplers.TPESampler(seed=SEED)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=MIN_EPOCHS)
study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner, study_name="notebook20_bayesopt_lstm")

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"Trial selesai: {len(completed)}/{N_TRIALS}  (dipangkas MedianPruner: {len(pruned)})")

best_trial = study.best_trial
best_artifact = trial_artifacts[best_trial.number]
best_config = best_artifact["config"]

print(f"\n=== Best trial #{best_trial.number} ===")
print(f"val_f1_macro = {best_trial.value:.4f}  (epoch terbaik = {best_artifact['best_epoch']}, threshold = {best_artifact['threshold']:.3f})")
print(json.dumps(best_config, indent=2))

## Bagian 4 — Diagnostik Proses Pencarian

Riwayat optimisasi (apakah `val_f1_macro` membaik seiring trial — tanda surrogate
model belajar area yang menjanjikan) dan *parameter importance* (hyperparameter mana
yang paling memengaruhi `val_f1_macro`, termasuk `hidden_size`/`num_layers` — jawaban
langsung atas pertanyaan "layer hidden yang cocok").

In [ ]:
trials_df = study.trials_dataframe()
print(trials_df[["number", "value", "state", "params_hidden_size", "params_num_layers"]].to_string(index=False))

fig1, ax1 = plt.subplots(figsize=(8, 4))
values = [t.value for t in study.trials if t.value is not None]
best_so_far = np.maximum.accumulate(values)
ax1.plot(range(1, len(values) + 1), values, "o", alpha=0.5, label="val_f1_macro per trial")
ax1.plot(range(1, len(best_so_far) + 1), best_so_far, "-", color="tab:red", label="Best so far")
ax1.set_xlabel("Trial ke-")
ax1.set_ylabel("val_f1_macro")
ax1.set_title("Riwayat Bayesian Optimization (Notebook 20)")
ax1.legend()
fig1.tight_layout()
fig1.savefig(OUT_DIR / "notebook20_optimization_history.png")
plt.show()

try:
    importances = optuna.importance.get_param_importances(study)
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    ax2.barh(list(importances.keys()), list(importances.values()))
    ax2.set_xlabel("Importance (fANOVA)")
    ax2.set_title("Kontribusi tiap hyperparameter terhadap val_f1_macro")
    ax2.invert_yaxis()
    fig2.tight_layout()
    fig2.savefig(OUT_DIR / "notebook20_param_importances.png")
    plt.show()
    print(json.dumps(importances, indent=2))
except (ValueError, RuntimeError) as e:
    # fANOVA butuh varians > 0 di tiap parameter & minimal beberapa trial selesai
    # (bisa gagal kalau trial completed terlalu sedikit / semua trial dipangkas awal).
    logger.warning("Param importance gagal dihitung: %s", e)

## Bagian 5 — Evaluasi Model Terbaik di Test Set

Model terbaik (state dict tersimpan di `trial_artifacts` selama Bayesian Optimization
berjalan — TIDAK retrain ulang) dievaluasi di `bundle.X_test`/`y_test` (15% terakhir,
belum pernah disentuh selama pencarian) memakai `best_artifact["threshold"]` (hasil
*sweep* di val, dipakai SEKALI ke test — sama pola dengan `run_notebook07_training()`
resmi, nol kebocoran metodologis).

In [ ]:
best_model = FlexibleLSTMClassifier(
    input_size=len(RETURN_FEATURE_COLS_B),
    hidden_size=best_config["hidden_size"],
    num_layers=best_config["num_layers"],
    dropout=best_config["dropout"],
    activation=best_config["activation"],
    num_classes=2,
).to(device)
best_model.load_state_dict(best_artifact["state_dict"])
best_model.eval()

X_test_t = torch.from_numpy(bundle.X_test).to(device)
with torch.no_grad():
    test_probs_up = torch.softmax(best_model(X_test_t), dim=1)[:, 1].cpu().numpy()
y_test_pred = (test_probs_up >= best_artifact["threshold"]).astype(int)

tn, fp, fn, tp = confusion_matrix(bundle.y_test, y_test_pred, labels=[0, 1]).ravel()
metrics_bo = {
    "accuracy": round(float(accuracy_score(bundle.y_test, y_test_pred)), 6),
    "precision": round(float(precision_score(bundle.y_test, y_test_pred, zero_division=0)), 6),
    "recall": round(float(recall_score(bundle.y_test, y_test_pred, zero_division=0)), 6),
    "f1": round(float(f1_score(bundle.y_test, y_test_pred, zero_division=0)), 6),
    "f1_macro": round(float(f1_score(bundle.y_test, y_test_pred, average="macro", zero_division=0)), 6),
    "recall_down": round(float(recall_score(bundle.y_test, y_test_pred, pos_label=0, zero_division=0)), 6),
    "recall_up": round(float(recall_score(bundle.y_test, y_test_pred, pos_label=1, zero_division=0)), 6),
}
confusion_bo = {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

print("Metrics (Bayesian Optimization, test set):")
print(json.dumps(metrics_bo, indent=2))
print("Confusion matrix:", confusion_bo)

## Bagian 6 — Perbandingan vs Random Search Resmi

Angka Random Search dibaca LANGSUNG dari metadata artefak resmi yang sudah tersimpan
(`finbert_lstm_skripsi_metadata.json`, hasil `notebook07_trainer.FINBERT_FILES`) —
BUKAN angka hardcode, supaya selalu sinkron dengan model resmi TERKINI. Kalau file
belum ada (Random Search resmi belum pernah dijalankan di environment ini), baris
tetap ditampilkan kosong apa adanya.

In [ ]:
official_metadata_path = Path(ARTIFACT_DIR) / FINBERT_FILES["metadata"]
official_meta = {}
if not official_metadata_path.is_absolute():
    official_metadata_path = ROOT / official_metadata_path
if official_metadata_path.exists():
    official_meta = json.loads(official_metadata_path.read_text(encoding="utf-8"))
else:
    logger.warning("Metadata Random Search resmi tidak ditemukan di %s", official_metadata_path)

rows = []
if official_meta:
    m, cfg = official_meta["metrics"], official_meta["config"]
    rows.append({
        "Metode": "Random Search (resmi, notebook07)",
        "hidden_size": cfg["hidden_size"], "num_layers": cfg["num_layers"],
        "dropout": round(cfg["dropout"], 3), "optimizer": cfg["optimizer"],
        "activation": cfg["activation"], "lr": cfg["lr"], "batch_size": cfg["batch_size"],
        "Accuracy": m["accuracy"], "Precision": m["precision"], "Recall": m["recall"],
        "F1": m["f1"], "F1_macro": m.get("f1_macro"),
    })
else:
    rows.append({"Metode": "Random Search (resmi, notebook07)", **{k: None for k in
                 ["hidden_size", "num_layers", "dropout", "optimizer", "activation", "lr",
                  "batch_size", "Accuracy", "Precision", "Recall", "F1", "F1_macro"]}})

rows.append({
    "Metode": "Bayesian Optimization (notebook 20)",
    "hidden_size": best_config["hidden_size"], "num_layers": best_config["num_layers"],
    "dropout": round(best_config["dropout"], 3), "optimizer": best_config["optimizer"],
    "activation": best_config["activation"], "lr": best_config["lr"], "batch_size": best_config["batch_size"],
    "Accuracy": metrics_bo["accuracy"], "Precision": metrics_bo["precision"], "Recall": metrics_bo["recall"],
    "F1": metrics_bo["f1"], "F1_macro": metrics_bo["f1_macro"],
})

comparison_df = pd.DataFrame(rows)
comparison_path = OUT_DIR / "notebook20_bayesopt_vs_randomsearch.csv"
comparison_df.to_csv(comparison_path, index=False)
print(comparison_df.to_string(index=False))
print(f"\nDisimpan ke: {comparison_path}")

## Bagian 7 — Simpan Artefak (terisolasi, TIDAK menimpa model resmi)

Disimpan ke `notebooks/outputs/notebook20/` (bukan `data/artifacts/lstm_weights/`) —
notebook ini murni eksperimen pembanding teknik pencarian, bukan pengganti model resmi
(`finbert_lstm_skripsi.*` TIDAK disentuh).

In [ ]:
torch.save({
    "model_state_dict": best_artifact["state_dict"],
    "config": best_config,
    "feature_cols": RETURN_FEATURE_COLS_B,
    "window": WINDOW,
    "horizon": RETURN_HORIZON,
    "threshold": best_artifact["threshold"],
}, OUT_DIR / "notebook20_bayesopt_best.pt")
joblib.dump(bundle.scaler, OUT_DIR / "notebook20_bayesopt_scaler.joblib")

trials_df.to_csv(OUT_DIR / "notebook20_optuna_trials.csv", index=False)

metadata = {
    "method": "bayesian_optimization_tpe",
    "n_trials": N_TRIALS,
    "n_completed": len(completed),
    "n_pruned": len(pruned),
    "search_space": SEARCH_SPACE,
    "best_config": best_config,
    "best_epoch": best_artifact["best_epoch"],
    "threshold": best_artifact["threshold"],
    "best_val_f1_macro": round(float(best_trial.value), 6),
    "metrics": metrics_bo,
    "confusion_matrix": confusion_bo,
    "feature_cols": RETURN_FEATURE_COLS_B,
    "window": WINDOW,
    "horizon": RETURN_HORIZON,
    "split_sizes": bundle.split_sizes,
    "eval_split": "test",
    "split_ratio": {"train": bundle.train_ratio, "val": bundle.val_ratio},
}
with open(OUT_DIR / "notebook20_bayesopt_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Artefak tersimpan di: {OUT_DIR}")

## Penutup

- **Belum dieksekusi** — `N_TRIALS=20` (masing-masing hingga `MAX_EPOCHS=80` dengan
  *early stopping*) perlu dijalankan terpisah; pantau `study.trials_dataframe()` untuk
  progres.
- Kalau Bayesian Optimization ternyata menemukan `val_f1_macro`/metrik test yang HANYA
  sedikit lebih baik (atau malah lebih buruk) dari Random Search resmi, itu tetap
  TEMUAN VALID — jangan menjalankan ulang dengan seed berbeda sampai "menang", itu
  p-hacking (prinsip yang sama dipegang di seluruh notebook eksperimen project ini).
  Klaim "Bayesian Optimization lebih baik dari Random Search" secara statistik butuh
  banyak repetisi seed pada KEDUA metode, bukan 1 vs 1 run.
- Bagian 4 (`param_importances`) menjawab langsung "hyperparameter mana yang paling
  menentukan performa" — kalau `hidden_size`/`num_layers` ternyata importance-nya
  RENDAH dibanding mis. `lr`/`dropout`, itu artinya ukuran hidden layer tidak terlalu
  sensitif untuk data ini (arsitektur kecil sudah cukup) — baca apa adanya, jangan
  dipaksa menyimpulkan "hidden layer besar lebih baik" kalau datanya tidak menunjukkan itu.
- Model hasil notebook ini TIDAK menggantikan model resmi (`finbert_lstm_skripsi.*`) —
  kalau hasilnya lebih baik dan ingin dipakai resmi, perlu keputusan eksplisit
  (menimpa artefak resmi via training manual/retrain, bukan otomatis dari notebook ini).